# HighRes-Net Inference with Diagnostics

This notebook provides comprehensive diagnostics to debug model output quality and image loading issues.

## 1. Setup and Imports

In [ ]:
import sys
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import ndimage

# Add src directory to path
sys.path.insert(0, '../src')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from DataLoader import TiffPatchDataset, collateFunction
from DeepNetworks.HRNet import HRNet
from diagnostics import (
    check_image_loading, 
    check_model_weights,
    compute_metrics,
    compare_with_bicubic
)

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Quick Diagnostic: Check Image Loading First

In [ ]:
# Verify TIFF dataset structure (new format, not old PNG format)
dataset_root = Path("D:\\GUC\\Datasets\\HighRes input test")
image_dirs = sorted([d for d in dataset_root.iterdir() if d.is_dir() and d.name.startswith('image_')])

print("\n" + "="*70)
print("DATASET VERIFICATION (TIFF Format)")
print("="*70)
print(f"\nDataset root: {dataset_root}")
print(f"Found {len(image_dirs)} image folders")

diag_results = {}

if len(image_dirs) > 0:
    diag_results['num_images'] = len(image_dirs)
    print(f"✓ Image folders: {[d.name for d in image_dirs[:5]]}{'...' if len(image_dirs) > 5 else ''}")
    
    # Check first image folder
    first_image = image_dirs[0]
    print(f"\nChecking first folder: {first_image.name}")
    
    lr_files = sorted(first_image.glob("LR_*.*"))
    hr_files = sorted(first_image.glob("HR.*"))
    
    print(f"  LR frames found: {len(lr_files)}")
    print(f"  HR files found: {len(hr_files)}")
    
    if len(lr_files) > 0 and len(hr_files) > 0:
        print(f"  ✓ Structure looks good (LR frames + HR present)")
        diag_results['structure_valid'] = True
        
        # Get file extensions
        lr_ext = lr_files[0].suffix
        hr_ext = hr_files[0].suffix
        print(f"  File types: LR{lr_ext}, HR{hr_ext}")
    else:
        print(f"  ✗ Missing LR frames or HR file!")
        diag_results['structure_valid'] = False
else:
    print(f"✗ No image_* folders found!")
    diag_results['num_images'] = 0

print("="*70)

## 3. Check Model Weights Status

In [ ]:
# Check if model has trained weights
weight_status = check_model_weights(Path("../models/weights/HRNet.pth"))

if not weight_status.get('has_weights', False):
    print("\n" + "="*70)
    print("⚠️ CRITICAL: Model is using RANDOM weights")
    print("="*70)
    print("""
This explains why your SR output looks bad:
- Grainy/noisy (random noise from uninitialized weights)
- Darker than expected (random negative scales)
- Artifacts/scan lines (random filter patterns)

NEXT STEPS:
1. Train the model on your data
2. Or find pre-trained weights for your domain
3. Or use bicubic upsampling for baseline comparison
    """)

## 4. Load Configuration

In [ ]:
config_path = '../config/config.json'
with open(config_path, 'r') as f:
    config = json.load(f)

print("Configuration:")
print(f"  n_views: {config['training']['n_views']}")
print(f"  min_L: {config['training']['min_L']}")
print(f"  batch_size: {config['training']['batch_size']}")
print(f"  Upsampling stride: {config['network']['decoder']['deconv']['stride']}x")
print(f"  Lambda range: {config['training'].get('lambda_range', 'Not set')}")

# Note: Scale validation happens during actual data loading below, 
# not from the diagnostic diag_results

## 5. Load Model Architecture

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = HRNet(config['network'])
model.to(device)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Model total parameters: {total_params:,}")
print(f"Model is in EVAL mode (no batch norm, no dropout)")

## 6. Load Pre-trained Weights

In [ ]:
weights_path = Path("../models/weights/HRNet.pth")

if weights_path.exists():
    print(f"Loading weights from {weights_path}")
    try:
        state_dict = torch.load(str(weights_path), map_location=device)
        model.load_state_dict(state_dict)
        print("✓ Weights loaded successfully!")
    except Exception as e:
        print(f"✗ Failed to load weights: {e}")
        print(f"  Using random initialization (output will be low quality)")
else:
    print(f"✗ Weights file not found at {weights_path}")
    print(f"  Using RANDOM initialization")
    print(f"  Expected output quality: POOR (grainy, noisy, artifacts)")

## 7. Create Dataset and DataLoader

In [ ]:
dataset_root = Path("D:\\GUC\\Datasets\\HighRes input test")
image_dirs = sorted([d for d in dataset_root.iterdir() if d.is_dir() and d.name.startswith('image_')])

print(f"Found {len(image_dirs)} images")
if len(image_dirs) > 0:
    print(f"Images: {[d.name for d in image_dirs[:5]]}...")

# Convert to strings
image_dirs_str = [str(d) for d in image_dirs]

# Create dataset
dataset = TiffPatchDataset(
    patch_dirs=image_dirs_str,
    max_views=config['training']['n_views']
)

print(f"Dataset size: {len(dataset)} images")

# Create dataloader
min_L = config['training']['min_L']
batch_size = config['training']['batch_size']

dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    collate_fn=collateFunction(min_L=min_L),
    pin_memory=torch.cuda.is_available()
)

print(f"DataLoader created with batch_size={batch_size}")

## 8. Run Inference on First Batch

In [ ]:
batch = next(iter(dataloader))
lrs, alphas, hrs, hr_maps, names = batch

print(f"Batch loaded:")
print(f"  LR shape: {lrs.shape}")
print(f"  Alphas shape: {alphas.shape}")
print(f"  HR shape: {hrs.shape}")
print(f"  HR_map shape: {hr_maps.shape}")
print(f"  Image names: {names}")

lrs = lrs.float().to(device)
alphas = alphas.float().to(device)

print(f"\nData moved to {device}")

## 9. Forward Pass with GPU Monitoring

**IMPORTANT:** Run all cells in order from top to bottom. Each cell depends on variables from previous cells.
If you get `NameError`, restart the kernel and re-run from the beginning without skipping cells.

In [ ]:
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    mem_before = torch.cuda.memory_allocated() / 1e9
    print(f"GPU memory before: {mem_before:.2f} GB")

with torch.no_grad():
    sr_output = model(lrs, alphas)

if torch.cuda.is_available():
    mem_after = torch.cuda.memory_allocated() / 1e9
    mem_peak = torch.cuda.max_memory_allocated() / 1e9
    print(f"GPU memory after: {mem_after:.2f} GB")
    print(f"Peak GPU memory: {mem_peak:.2f} GB")

print(f"\nSR output shape: {sr_output.shape}")
print(f"SR output dtype: {sr_output.dtype}")
print(f"SR value range: [{sr_output.min():.4f}, {sr_output.max():.4f}]")

In [ ]:
print("\n" + "="*70)
print("OUTPUT RANGE ANALYSIS - Check for Range Regularization Issues")
print("="*70)

sr_batch_min = sr_output.min().item()
sr_batch_max = sr_output.max().item()
sr_batch_range = sr_batch_max - sr_batch_min

hrs_batch_min = hrs.min().item()
hrs_batch_max = hrs.max().item()
hrs_batch_range = hrs_batch_max - hrs_batch_min

print(f"\n📊 SR Output (from model):")
print(f"  Min: {sr_batch_min:.4f}, Max: {sr_batch_max:.4f}")
print(f"  Range: {sr_batch_range:.4f}")
print(f"  Mean: {sr_output.mean():.4f}, Std: {sr_output.std():.4f}")

print(f"\n📊 HR Ground Truth:")
print(f"  Min: {hrs_batch_min:.4f}, Max: {hrs_batch_max:.4f}")
print(f"  Range: {hrs_batch_range:.4f}")
print(f"  Mean: {hrs.mean():.4f}, Std: {hrs.std():.4f}")

print(f"\n🔍 ANALYSIS:")

target_sr_range = 0.6  # What range regularization targets

if sr_batch_range < 0.3:
    print(f"  🔴 RANGE SEVERELY COMPRESSED: {sr_batch_range:.4f} < 0.3")
    print(f"     This is causing BLUR! The network learns conservative middle values")
    print(f"     Fix: Reduce lambda_range in config.json")
    print(f"     Current: 0.02 → Try: 0.005 or 0.01")
elif sr_batch_range < hrs_batch_range * 0.7:
    print(f"  🟠 RANGE COMPRESSED: {sr_batch_range:.4f} vs HR {hrs_batch_range:.4f}")
    print(f"     Output is less dynamic than ground truth (likely blurry)")
    print(f"     Fix: Reduce lambda_range slightly")
    print(f"     Current: 0.02 → Try: 0.01")
elif sr_batch_range > hrs_batch_range * 1.3:
    print(f"  🟠 RANGE TOO WIDE: {sr_batch_range:.4f} vs HR {hrs_batch_range:.4f}")
    print(f"     Network forced extreme values (noisy output)")
    print(f"     Fix: Increase lambda_range")
    print(f"     Current: 0.02 → Try: 0.05")
else:
    print(f"  ✓ RANGE GOOD: SR {sr_batch_range:.4f} matches HR {hrs_batch_range:.4f}")
    print(f"    Range loss is working correctly")

# Check contrast ratio
sr_std = sr_output.std().item()
hrs_std = hrs.std().item()
std_ratio = sr_std / hrs_std if hrs_std > 0 else 0

print(f"\n📈 Contrast (std deviation):")
print(f"  SR/HR ratio: {std_ratio:.3f}")

if std_ratio < 0.7:
    print(f"  ⚠️  SR has LESS contrast than HR → BLUR indicator")
    print(f"     Reduce lambda_range to let network produce more detail")
elif std_ratio > 1.3:
    print(f"  ⚠️  SR has MORE contrast than HR → NOISE indicator")
    print(f"     Increase lambda_range to regularize extremes")
else:
    print(f"  ✓ Contrast ratio looks good")

print("="*70)

## 10. Extract and Analyze First Sample

In [ ]:
sr = sr_output[0, 0].cpu().numpy()
lr_first = lrs[0, 0].cpu().numpy()
hr_true = hrs[0].numpy() if torch.is_tensor(hrs) else hrs[0] if isinstance(hrs, list) else None

# Squeeze channel dimension if present
if hr_true is not None and hr_true.ndim == 3:
    hr_true = np.squeeze(hr_true, axis=0)

# Create upsampled LR for visual comparison
lr_upsampled = np.repeat(np.repeat(lr_first, 2, axis=0), 2, axis=1)
sr = np.clip(sr, 0, 1)

print(f"Extracted first sample:")
print(f"  LR: {lr_first.shape} → SR: {sr.shape} → HR: {hr_true.shape if hr_true is not None else 'Not loaded'}")

## 11. Compute Quality Metrics

In [ ]:
from scipy import ndimage

print("\n" + "="*70)
print("Image Quality Metrics (SR vs Ground Truth)")
print("="*70)

if hr_true is not None:
    # Ensure hr_true is 2D
    if hr_true.ndim == 3:
        hr_true = np.squeeze(hr_true, axis=0)
    
    # Resize HR to match SR if needed
    if hr_true.shape != sr.shape:
        scale = sr.shape[0] / hr_true.shape[0]
        hr_true = ndimage.zoom(hr_true, scale, order=3)
    
    # Compute metrics for HighRes-Net
    nn_metrics = compute_metrics(hr_true, sr)
    if nn_metrics['valid']:
        print(f"\n🧠 HighRes-Net Output:")
        print(f"  PSNR: {nn_metrics['psnr']:.2f} dB ({nn_metrics['psnr_label']})")
        print(f"  SSIM: {nn_metrics['ssim']:.4f} ({nn_metrics['ssim_label']})")
    else:
        print(f"Cannot compute metrics: {nn_metrics['reason']}")
    
    # Compute bicubic baseline
    lr_bicubic = ndimage.zoom(lr_first, 2, order=3)
    if lr_bicubic.shape != hr_true.shape:
        scale = hr_true.shape[0] / lr_bicubic.shape[0]
        lr_bicubic = ndimage.zoom(lr_bicubic, scale, order=3)
    
    bicubic_metrics = compute_metrics(hr_true, lr_bicubic)
    if bicubic_metrics['valid']:
        print(f"\n📊 Bicubic Baseline:")
        print(f"  PSNR: {bicubic_metrics['psnr']:.2f} dB ({bicubic_metrics['psnr_label']})")
        print(f"  SSIM: {bicubic_metrics['ssim']:.4f} ({bicubic_metrics['ssim_label']})")
        
        if nn_metrics['valid']:
            psnr_diff = nn_metrics['psnr'] - bicubic_metrics['psnr']
            print(f"\n📈 HRNet vs Bicubic: PSNR {psnr_diff:+.2f} dB", 
                  "✓" if psnr_diff > 0 else "✗")
else:
    print("HR ground truth not available")

print("="*70)

## 12. Visualize Comparison

In [ ]:
if hr_true is not None:
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    
    # Top left: LR upsampled (nearest neighbor)
    axes[0, 0].imshow(lr_upsampled, cmap='gray')
    axes[0, 0].set_title('LR View (2x nearest-neighbor)')
    axes[0, 0].axis('off')
    
    # Top right: LR bicubic
    lr_bicubic = ndimage.zoom(lr_first, 2, order=3)
    axes[0, 1].imshow(lr_bicubic, cmap='gray')
    axes[0, 1].set_title('LR (2x bicubic)')
    axes[0, 1].axis('off')
    
    # Bottom left: HighRes-Net SR
    axes[1, 0].imshow(sr, cmap='gray')
    axes[1, 0].set_title('HighRes-Net SR Output')
    axes[1, 0].axis('off')
    if not weight_status.get('has_weights', False):
        axes[1, 0].text(0.5, 0.02, '(Random weights)', 
                        ha='center', transform=axes[1, 0].transAxes,
                        fontsize=10, color='red', weight='bold')
    
    # Bottom right: HR ground truth
    axes[1, 1].imshow(hr_true, cmap='gray')
    axes[1, 1].set_title('HR Ground Truth')
    axes[1, 1].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("Cannot visualize: HR not loaded")

## 13. Summary & Recommendations

In [ ]:
print("\n" + "="*70)
print("DIAGNOSTIC SUMMARY")
print("="*70)

if weight_status.get('has_weights', False):
    print("✓ Model weights loaded successfully")
else:
    print("⚠️  Model using RANDOM weights (untrained)")
    print("   Run training_run.ipynb to train the model")

print("\n✅ Next: Check PSNR above")
print("   - If PSNR >> bicubic: Model is working well")
print("   - If PSNR ≈ bicubic: Model needs more training")
print("="*70)